# Anima 引擎部署（Sprint 10：Worker API 轮询 + LLM 4 槽位容灾）

前置：Cloudflare Worker 已部署并配置 `ENGINE_KEY` secret。
本 notebook 在下方单元格填写 ①你的仓库地址（含改造后代码） ②Worker 地址 ③ENGINE_KEY ④LLM 4 槽位配置（第 1 个为主，出错按序切换；共用同一 model）。

In [ ]:
# ===== 必填配置（已写死；后续有变化请直接改这里） =====
REPO_URL = "https://github.com/Reaky-Dawn/AnimaBot.git"  # 你的引擎代码仓库
WORKER_BASE_URL = "https://anima-web.chenzilong315.workers.dev"  # Worker 域名（animadraw.cloud 生效后改为 https://animadraw.cloud）
ENGINE_KEY = "LFP4SoGh6O1Xb5nxQzT3fItkjegwVmsaicEMWDJHCvpNduKZ"  # 与 Worker `wrangler secret put ENGINE_KEY` 一致
ENGINE_ID = "engine-1"

# ===== LLM 配置（Sprint 10：4 个槽位，第 1 个为主，出错按序切换；共用同一模型） =====
# 每个槽位只需 api_key + base_url；model 全局一个（如 deepseek-v4-flash）
MODEL = "deepseek-v4-flash"
PROVIDERS = [
    {"api_key": "", "base_url": ""},
    {"api_key": "", "base_url": ""},
    {"api_key": "", "base_url": ""},
    {"api_key": "", "base_url": ""},
]

import os, json, subprocess, sys, time
os.environ["WORKER_BASE_URL"] = WORKER_BASE_URL
os.environ["ENGINE_KEY"] = ENGINE_KEY
os.environ["ENGINE_ID"] = ENGINE_ID

# 启动前校验（已写死，但保留检查以防意外）
assert WORKER_BASE_URL not in ("", "http://127.0.0.1:8787", "https://anima.example.com"), \
    f"WORKER_BASE_URL 仍为占位符（当前={WORKER_BASE_URL}），请修改"
if len(ENGINE_KEY) < 16:
    print('⚠️ 警告：ENGINE_KEY 长度异常，请检查')
print("配置已设置（WORKER_BASE_URL=", WORKER_BASE_URL, "，ENGINE_KEY 长度:", len(ENGINE_KEY), "，LLM 槽位数:", len(PROVIDERS), "）")

In [ ]:
!git clone https://github.com/chinokikiss/ComfyUI.git || true
!rm -rf /kaggle/working/AnimaBot && git clone {REPO_URL} /kaggle/working/AnimaBot

In [ ]:
# ===== 模型复制（Sprint 11：从 Kaggle input 复制本地模型，不再从 HuggingFace 下载） =====
# 模型经 Kaggle 的 Add Input -> 连接你的 dataset 挂载到 /kaggle/input/<dataset名>/ 下。
# 这里在整个 /kaggle/input 下递归搜索每个目标模型并复制到 ComfyUI 对应目录。
import os, shutil, pathlib, zipfile

INPUT_DIR = pathlib.Path('/kaggle/input')
if not INPUT_DIR.exists() or not any(INPUT_DIR.iterdir()):
    print('⚠️ 未在 /kaggle/input 找到任何输入！请先 Add Input 连接你的 dataset（含模型）。'
          '引擎启动后可能因模型缺失而失败，但会话不会被终止。')

targets = {
    'diffusion_models': ['miaomiaoHarem_anima12.safetensors'],
    'text_encoders': ['miaomiaoHarem_anima14_txt.safetensors'],
    'vae': ['qwenImage_qwenImageVAE.safetensors'],
    'upscale_models': ['4x-AnimeSharp.safetensors'],
}

def find_model(name):
    """在整个 /kaggle/input 递归找目标模型文件；也支持 zip 内（解压前先找 zip）。"""
    for f in INPUT_DIR.rglob(name):
        if f.is_file():
            return f
    # 若模型在 zip 压缩包里，先解压到临时目录再找
    for z in INPUT_DIR.rglob('*.zip'):
        try:
            with zipfile.ZipFile(z) as zf:
                if name in zf.namelist():
                    tmp = pathlib.Path('/kaggle/working/_models_unzip')
                    zf.extract(name, tmp)
                    return tmp / name
        except Exception:
            continue
    return None

missing = []
for subdir, names in targets.items():
    dest_dir = pathlib.Path('/kaggle/working/ComfyUI/models') / subdir
    dest_dir.mkdir(parents=True, exist_ok=True)
    for name in names:
        src = find_model(name)
        if src is None:
            missing.append(name)
            continue
        dst = dest_dir / name
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            shutil.copy2(src, dst)
            print(f'已复制: {name} -> ComfyUI/models/{subdir}/')
        else:
            print(f'已存在跳过: {name}')

if missing:
    print('⚠️ 缺失以下模型（引擎启动后可能因模型缺失而失败，但不会杀会话）:', missing)
else:
    print('模型复制完成')

In [ ]:
!pip install -r /kaggle/working/ComfyUI/requirements.txt -q
!pip install -r /kaggle/working/AnimaBot/requirements.txt -q
!pip install sageattention -q
!apt-get install -y oxipng > /dev/null 2>&1 || echo "oxipng 安装失败（可选，跳过压缩）"

In [ ]:
import json
cfg = {
  "providers": PROVIDERS,
  "model": MODEL,
  "logging": True,
}
with open("/kaggle/working/AnimaBot/config.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)
print("config.json 已写入（providers:", len(PROVIDERS), "）")

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Sprint 11.5：不再显式启动 ComfyUI——引擎 core.py 自管理：
#   空闲 IDLE_TIMEOUT_SEC 秒无任务 -> 关闭 ComfyUI 释放 GPU（零消耗）
#   有新任务 -> 冷启动 ComfyUI（加载模型约 30-60s），用户需稍等
# 引擎只启动 core.py + 保活阻塞。
log_dir = Path('/kaggle/working/engine_logs')
log_dir.mkdir(exist_ok=True)

base_env = dict(os.environ)
base_env.update({
    "WORKER_BASE_URL": WORKER_BASE_URL,
    "ENGINE_KEY": ENGINE_KEY,
    "ENGINE_ID": ENGINE_ID,
    "IDLE_TIMEOUT_SEC": "300",  # 空闲 5 分钟关 ComfyUI；可改为 60 更快释放
})

out = open(str(log_dir / "engine.log"), "w")
err = open(str(log_dir / "engine.err"), "w")
p = subprocess.Popen([sys.executable, "-u", "/kaggle/working/AnimaBot/core.py"],
                     cwd="/kaggle/working/AnimaBot", env=base_env, stdout=out, stderr=err)
print("已启动引擎 core.py PID", p.pid)

print("\n引擎已启动（ComfyUI 由引擎按需冷启动，空闲自动休眠释放 GPU）")
print("引擎日志: /kaggle/working/engine_logs/engine.log （tail -u 查看实时）")
print("错误明细: /kaggle/working/engine_logs/errors.log （每次任务失败追加完整步骤日志）")

# ===== 保活阻塞：让 commit 任务永不「执行完」 =====
# Save & Run All (commit) 执行完所有单元格后 Kaggle 会判定任务完成并回收容器。
# 加入下方 while True 循环后，commit 任务永远阻塞在这里，不返回 -> 容器保持 -> 引擎子进程持续跑。
# 直到 12h 免费版上限到点，Kaggle 自动终止。
import time
print("进入保活模式。此 commit 任务将持续占用容器（约 12h 上限）。")
try:
    while True:
        time.sleep(600)  # 每 10 分钟循环一次，维持容器活跃
except KeyboardInterrupt:
    print("保活结束")